In [1]:
%%configure -f
{
    "conf": {
        "spark.jars": "s3://ne-prod-data-pipeline-orchestrator/elasticmapreduce/external-jars/delta-spark.jar ,s3://ne-prod-delta-lake-secure-layer/elasticmapreduce/external-jars/encrypt-decrypt-spark-1.0-SNAPSHOT.jar",
        "spark.submit.pyFiles": "s3://ne-prod-data-pipeline-orchestrator/elasticmapreduce/external-jars/delta-spark.jar,s3://ne-prod-data-pipeline-orchestrator/ds-da-repo/emr_spark_utils.zip"
    }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
628,application_1779081930764_0620,pyspark,idle,Link,Link,assumed-role_AWSReservedSSO_ne-prod-analytics-da-sso_3470bf0916eed697_shlok_suraiya_slicebank_com,


In [2]:
from decimal import Decimal
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import openpyxl as xl
from openpyxl.utils import get_column_interval
from openpyxl.utils.dataframe import dataframe_to_rows
import re
import json
import operator
import boto3
import email
import smtplib
import ssl
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
import time
import enum
from pyspark.sql import SparkSession
import math
import io
from pyspark.sql import functions as F
import pytz
import requests

from pyspark.sql.functions import col, split, get_json_object, date_sub, current_timestamp, max as spark_max, min as spark_min, collect_list, slice
from pyspark.sql.types import DoubleType

from pyspark.sql.functions import col, split, get_json_object, current_date, date_sub, year, month, dayofmonth, broadcast, expr, sort_array, struct
from pyspark.sql.types import DoubleType

from pyspark.sql.functions import explode, row_number

from pyspark.sql.window import Window
# Compute yesterday and today
yesterday = date_sub(current_date(), 1)
today = current_date()


s3 = boto3.client('s3')
date_yesterday = datetime.now() - timedelta(days=1)

date_yesterday_str = date_yesterday.strftime('%Y-%m-%d')

def get_secrets(secret_name):
    region_name = "ap-south-1"
    session = boto3.session.Session()

    client = session.client(service_name="secretsmanager", region_name=region_name)

    try:
        get_secret_value_response = client.get_secret_value(SecretId=secret_name)

    except ClientError as e:
        if e.response.get("Error").get("Code") == "ResourceNotFoundException":
            raise Exception("The requested secret " + secret_name + " was not found")
        elif e.response.get("Error").get("Code") == "InvalidRequestException":
            raise Exception("The request was invalid due to:", e)
        else:
            raise Exception("The request failed because of:", e)

    secret = get_secret_value_response.get("SecretString")

    if isinstance(secret, str):
        secret = eval(secret)

    return secret

MSSQL_ANALYTICS_CLUSTER_COMMON = "prod/data/analytics/cbs/common" #verified
BANKOS = "prod/data/analytics/bankos"
MIS_8004 = "prod/data/analytics/cbs/mis_8004"
MIS_2397 = "prod/data/analytics/cbs/mis_2397"
BANK_KYC = "prod/data/analytics/bank_kyc"
BANK_KYC_RW = "prod/data/analytics/bankkyc_rw"
BANK_ONBOARDING_DB = "prod/data/analytics/bank_onboarding"
BANK_ONBOARDING_DB_RW = "prod/data/analytics/bank_onboarding_rw"
COMMON_TOKEN = 'prod/data/analytics/common-token'
SUNRISE_MAIL = 'prod/data/analytics/sunrise-mail'
BANK_PAYMENT_LINKS_ACCESS_TOKEN_V2  = 'prod/data/analytics/bank-payment-links-access-token-v2'
BANK_PAYMENT_LINKS_ACCESS_TOKEN_V3  = 'prod/data/analytics/bank-payment-links-access-token-v3'
RAZORPAY = 'prod/data/analytics/cbs/razorpay'
HELO_AI = 'prod/data/analytics/heloai'
SERVICE_TOKEN = 'prod/data/analytics/service-token'
CONFLUENT_TOKEN = 'prod/data/analytics/confluent-token'

    
from slack_sdk import WebClient
from slack_sdk.errors import SlackApiError
from datetime import datetime
slack_token =get_secrets(COMMON_TOKEN)["slack_token"]
client = WebClient(token=slack_token)

spark.udf.registerJavaFunction("decryptFunction", "com.piixie.SparkDecrypt")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

from pyspark.sql.functions import split, col, udf , lit
from pyspark.sql.types import DoubleType
import pygeohash as pgh
def get_geohash(lat, long, precision):
    if lat is None : 
        lat = 0.0000
    else : 
        lat = lat
    if long is None : 
        long = 0.0000
    else :
        long = long
    return pgh.encode(lat, long, precision)

udf_get_geohash = udf(get_geohash)

def push_to_slack(df1):
    channel_id = "C0958LSLM8Q"
    df1.to_csv('./sample.csv', index=False)
    if df1.shape[0] > 0 : 
        client.files_upload_v2(
        channel = channel_id,
        title = "df_base",
        file = "./sample.csv",
        initial_comment="df_base",
        )
        print(f'pushed to slack: {channel_id}')

def load_from_s3(path, view_name):
    data = spark.read.parquet(path)
    data.createOrReplaceTempView(view_name)
    
    return data
    
def load_csv(path, view_name):    
    import pandas as pd
    import io
    import boto3

    # Parse S3 path
    path = path.replace("s3://", "")
    bucket_name = path.split("/")[0]
    key = "/".join(path.split("/")[1:])

    # Read from S3 using boto3
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket_name, Key=key)

    # Read Excel with pandas
    data = io.BytesIO(obj['Body'].read())
    df = pd.read_csv(data)

    # Convert to Spark DataFrame
    df = spark.createDataFrame(df)

    # Create temporary view
    df.createOrReplaceTempView(view_name)

def load_excel(path, view_name):
    path = path.replace("s3://", "")
    bucket_name = path.split("/")[0]
    key = "/".join(path.split("/")[1:])

    # Read from S3 using boto3
    s3 = boto3.client('s3')
    obj = s3.get_object(Bucket=bucket_name, Key=key)

    # Read Excel with pandas
    excel_data = io.BytesIO(obj['Body'].read())
    df = pd.read_excel(excel_data)
    df.columns = df.columns.str.strip()

    # Convert to Spark DataFrame
    blocked_df = spark.createDataFrame(df)
    blocked_df.createOrReplaceTempView(view_name)
    
    return blocked_df

from pyspark.sql.functions import split, col, udf , lit
from pyspark.sql.types import DoubleType
import pygeohash as pgh
def get_geohash(lat, long, precision):
    if lat is None : 
        lat = 0.0000
    else : 
        lat = lat
    if long is None : 
        long = 0.0000
    else :
        long = long
    return pgh.encode(lat, long, precision)

udf_get_geohash = udf(get_geohash)

######### TOKEN SET RATIO

from pyspark.sql.functions import udf, col, when
from pyspark.sql.types import IntegerType
import re
from difflib import SequenceMatcher

def tokenize(s):
    if s is None:
        return []
    s = re.sub(r'[^a-zA-Z0-9 ]', '', s.lower())
    return s.split()

def ratio(s1, s2):
    return int(SequenceMatcher(None, s1, s2).ratio() * 100)

def token_set_ratio(s1, s2):
    if s1 is None or s2 is None:
        return 0
    
    tokens1 = set(tokenize(s1))
    tokens2 = set(tokenize(s2))
    
    common_tokens = tokens1.intersection(tokens2)
    diff1 = tokens1.difference(tokens2)
    diff2 = tokens2.difference(tokens1)
    
    sorted_common = ' '.join(sorted(common_tokens))
    sorted_diff1 = ' '.join(sorted(diff1))
    sorted_diff2 = ' '.join(sorted(diff2))
    
    # Combine strings in three ways and calculate ratio
    combined1 = sorted_common + ' ' + sorted_diff1
    combined2 = sorted_common + ' ' + sorted_diff2
    
    ratios = [
        ratio(sorted_common, combined1),
        ratio(sorted_common, combined2),
        ratio(combined1, combined2),
    ]
    
    return max(ratios)


token_set_ratio_udf = udf(token_set_ratio, IntegerType())

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
629,application_1779081930764_0621,pyspark,idle,Link,Link,assumed-role_AWSReservedSSO_ne-prod-analytics-da-sso_3470bf0916eed697_rayansh_khamesra_slicebank_com,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
spark.sql("""
SELECT 
    user_uuid, 
    src_txn_id,
    narration,
    debit_party_type,
    credit_party_type
FROM casa_txn_gold.transaction
WHERE year = 2026 --and month = 2 and user_uuid = '225e88ec-3809-4792-8e15-2d10d0012314' --and narration = 'Payment to slice merchant QR'
""").createOrReplaceTempView('casa_data')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
df = spark.sql("""
SELECT 
    a.*, 
    b.account_open_date as onboarding_date,
    c.narration as casa_narration,
    c.debit_party_type,
    c.credit_party_type,
    d.note as upi_note
FROM cyber_crime_gold.final_view_deduped a
JOIN bsgcore_gold.account_master b on a.account_id = b.account_id and b.is_active = 1 and b.__is_deleted=False and b.product_code = 1150
LEFT JOIN casa_data c on a.src_txn_id = c.src_txn_id
LEFT JOIN upiswitch_tpap_gold.cbs_transactions d on d.year = 2026 and a.rrn = d.rrn
""").createOrReplaceTempView('dca_txn_data')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
dca_txn_df = spark.sql("""
SELECT a.*, b.dsa_account_holder_name as user_name
FROM dca_txn_data a
JOIN s3_tool_propagator_pii.dca_fraud_base_data b on a.uuid = b.uuid
""")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [6]:
dca_txn_df = spark.sql("""
SELECT a.*, b.dsa_account_holder_name as user_name
FROM dca_txn_data a
JOIN s3_tool_propagator_pii.dca_fraud_base_data b on a.uuid = b.uuid
""")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [7]:
dca_txn_df = dca_txn_df.withColumn("cp_name_match", token_set_ratio_udf(col("user_name"), col("counter_party_cbs_name")))

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [8]:
dca_txn_df.createOrReplaceTempView('dca_txn_data')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [9]:
spark.sql("""
    SELECT uuid, gst_ref_id,
           ROW_NUMBER() OVER (PARTITION BY uuid ORDER BY created_at DESC) AS rn
    FROM kycdb_gold.business_details
    WHERE __is_deleted = FALSE OR __is_deleted IS NULL
""").filter(F.col("rn") == 1).createOrReplaceTempView('gst_flag')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [10]:
spark.sql("""
with ifsc_lookup AS (
    SELECT DISTINCT ifsc, UPPER(TRIM(state)) AS state
    FROM s3_tool_propagator.ifsc_state_mapping
    WHERE state IS NOT NULL AND TRIM(state) <> ''
)
    SELECT
        t.rrn,
        'UPI' AS txn_rail,
        CAST(t.amount AS DOUBLE) AS amount,
        t.counter_party_ifsc,
        ifsc.state AS counterparty_state,
        b.kyc_state,
        CASE
            WHEN ifsc.state IS NOT NULL
             AND ifsc.state <> UPPER(TRIM(b.kyc_state))
            THEN 1 ELSE 0
        END AS is_cross_state
    FROM upiswitch_tpap_gold.cbs_transactions t
    JOIN s3_tool_propagator_pii.dca_fraud_base_data b on t.user_account_number = b.account_number
    LEFT JOIN ifsc_lookup ifsc ON t.counter_party_ifsc = ifsc.ifsc
    WHERE t.year >= 2026
      AND t.txn_status = 'SUCCESS'
      AND (t.__is_deleted IS NULL OR t.__is_deleted = false)
""").createOrReplaceTempView('upi_cross_state')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [11]:
spark.sql("""
with ifsc_lookup AS (
    SELECT DISTINCT ifsc, UPPER(TRIM(state)) AS state
    FROM s3_tool_propagator.ifsc_state_mapping
    WHERE state IS NOT NULL AND TRIM(state) <> ''
)
    SELECT
        rrn,
        'IMPS' AS txn_rail,
        CAST(t.amount AS DOUBLE) AS amount,
        t.counter_party_ifsc,
        ifsc.state AS counterparty_state,
        b.kyc_state,
        CASE
            WHEN ifsc.state IS NOT NULL
             AND ifsc.state <> UPPER(TRIM(b.kyc_state))
            THEN 1 ELSE 0
        END AS is_cross_state
    FROM imps_switch_gold.imps_transactions t
    JOIN imps_switch_gold.imps_transaction_statuses ts
        ON t.id = ts.txn_id AND ts.txn_status = 'SUCCESS'
    JOIN s3_tool_propagator_pii.dca_fraud_base_data b on t.user_account_number = b.account_number
    LEFT JOIN ifsc_lookup ifsc ON t.counter_party_ifsc = ifsc.ifsc
    WHERE t.year >= 2026
      AND (t.__is_deleted IS NULL OR t.__is_deleted = false)
""").createOrReplaceTempView('imps_cross_state')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
txn_df = spark.sql("""
SELECT 
    a.*,
    COALESCE(b.kyc_state, c.kyc_state) as kyc_state,    
    COALESCE(b.counterparty_state, c.counterparty_state) as counterparty_state,
    COALESCE(b.is_cross_state, c.is_cross_state) as is_cross_state
FROM dca_txn_data a
LEFT JOIN upi_cross_state b on a.rrn = b.rrn
LEFT JOIN imps_cross_state c on a.rrn = c.rrn
""").createOrReplaceTempView('dca_cross_state')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [13]:
spark.sql("""
WITH txn_length_data AS (
    SELECT
        *,
        LENGTH(CAST(CAST(txn_amount AS BIGINT) AS STRING)) AS digits,
        CAST(txn_amount AS BIGINT) AS amt
    FROM dca_cross_state
),

close_to_round_txn_data AS (
    SELECT
        *,
        CASE WHEN ABS(amt - (ROUND(amt / POWER(10, digits - 1)) * POWER(10, digits - 1))) <= POWER(10, digits - 3) * 5 THEN 1 ELSE 0 END AS round_txn_flag,
        CASE WHEN txn_amount != CAST(txn_amount AS BIGINT) THEN 1 ELSE 0 END AS decimal_txn_flag
    FROM txn_length_data
)

SELECT
    uuid,
    SUM(txn_amount) as txn_amount,
    count(txn_ref_no) as txn_count,
    --SUM(CASE WHEN RIGHT(CAST(txn_amount as bigint), 2) = '99' THEN txn_amount END) as 99_txn_amount,
    --COUNT(CASE WHEN RIGHT(CAST(txn_amount as bigint), 2) = '99' THEN txn_amount END) as 99_txn_count,
    --SUM(CASE WHEN RIGHT(CAST(txn_amount as bigint), 3) = '999' THEN txn_amount END) as 999_txn_amount,
    --COUNT(CASE WHEN RIGHT(CAST(txn_amount as bigint), 3) = '999' THEN txn_amount END) as 999_txn_count,
    --COUNT(DISTINCT CASE WHEN RIGHT(CAST(txn_amount as bigint), 3) = '999' THEN counter_party_cbs_name END) as 999_cp_count,
    --sum(CASE WHEN cp_name_match < 80 THEN txn_amount END) as mismatch_debit_amount,
    --count(CASE WHEN cp_name_match < 80 THEN txn_amount END) as mismatch_debit_count,
    --SUM(CASE WHEN round_txn_flag = 1 THEN txn_amount END) as round_txn_amount,
    --COUNT(CASE WHEN round_txn_flag = 1 THEN txn_amount END) as round_txn_count,
    --SUM(CASE WHEN decimal_txn_flag = 1 THEN txn_amount END) as decimal_txn_amount,
    --COUNT(CASE WHEN decimal_txn_flag = 1 THEN txn_amount END) as decimal_txn_count,
    SUM(CASE WHEN is_cross_state = 1 THEN txn_amount END) as cross_state_txn_amount,
    COUNT(CASE WHEN is_cross_state = 1 THEN txn_amount END) as cross_state_txn_count,
    COUNT(distinct counterparty_state) as cp_state_count,
    SUM(CASE WHEN lower(upi_note) like '%do%not%modify%' THEN txn_amount END) as dnm_txn_amount,
    COUNT(CASE WHEN lower(upi_note) like '%do%not%modify%' THEN txn_amount END) as dnm_txn_count
FROM close_to_round_txn_data
WHERE txn_date <= onboarding_date + interval '7' DAY AND txn_mode <> 'IFT - Interest Credits' and txn_nature = 'C' and narration <> 'Payment to slice merchant QR'
group by 1
""").createOrReplaceTempView('dca_analysis_data')

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [14]:
%%pretty
spark.sql("""
with state_level_credits as (
    SELECT 
        uuid, 
        counterparty_state, 
        sum(txn_amount) as txn_amount,
        count(distinct txn_ref_no) as txn_count
    FROM dca_cross_state
    WHERE 1=1
        and txn_date <= onboarding_date + interval '7' DAY 
        AND txn_mode <> 'IFT - Interest Credits' 
        and txn_nature = 'C' 
        and narration <> 'Payment to slice merchant QR'
        and kyc_state <> counterparty_state
    group by 1, 2
)
SELECT 
    a.uuid,
    count(distinct counterparty_state) as cp_state_count,
    count(distinct case when txn_amount > 5000 THEN counterparty_state END) as cp_state_5k_count,
    count(distinct case when txn_amount > 10000 THEN counterparty_state END) as cp_state_10k_count
FROM state_level_credits a
group by 1
""").createOrReplaceTempView('cp_state_data')

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [18]:
%%pretty
df = spark.sql("""
with mha_data as (
SELECT account_number_internal, min(reporting_date) as min_reporting_date, min(txn_date_mha) as min_mha_txn
FROM s3_tool_propagator_pii.mha_case_registry
WHERE txn_nature_internal = 'C' and layer in (0, 1)
group by 1
), 
freeze_data as (
SELECT 
    account_id,
    min(created_date) as first_freeze_date,
    min_by(remarks, created_date) as first_freeze_remark,
    max_by(remarks, created_date) as last_freeze_remark
FROM bsgcore_gold.account_multi_freeze_status 
WHERE 1=1
    and __is_deleted=False
    and is_active = 1
    and not freeze_status in (1, 9)
group by 1
)
SELECT 
    a.*,
    b.L1,
    b.reported,
    mha.min_reporting_date,
    mha.min_mha_txn,
    b.freeze_flag,
    b.combined_l1,
    b.signup_time,
    b.onboarding_date,
    MONTH(b.onboarding_date) as onb_month,
    b.itc_flag,
    b.uuid,
    b.kyc_state,
    b.udyam_date_of_registration,
    DATEDIFF(b.onboarding_date, b.udyam_date_of_registration) as udyam_to_onb,
    b.mcc,
    b.merchant_category,
    1 as users,
    c.gst_ref_id,
    f.decile as po_decile,
    f.score as po_score,
    g.decile as o_decile,
    g.model_score as o_score,
    h.cp_state_5k_count,
    h.cp_state_10k_count,
    i.first_freeze_date,
    i.first_freeze_remark,
    i.last_freeze_remark
FROM s3_tool_propagator_pii.dca_fraud_base_data b
JOIN dca_analysis_data a on a.uuid = b.uuid
LEFT JOIN gst_flag c on b.uuid = c.uuid

LEFT JOIN s3_tool_propagator_pii.post_onboarding_fraud_model_v1 f on a.uuid = f.uuid
LEFT JOIN s3_tool_propagator_pii.onboarding_fraud_model_scoring_v2 g on a.uuid = g.uuid
LEFT JOIN mha_data mha on b.account_number = mha.account_number_internal
LEFT JOIN cp_state_data h on a.uuid = h.uuid
LEFT JOIN freeze_data i on b.account_id = i.account_id
WHERE YEAR(b.onboarding_date) = 2026
AND MONTH(b.onboarding_date) = 6
AND h.cp_state_5k_count >= 8
AND LOWER(b.kyc_state) IN (
    'west bengal',
    'andhra pradesh',
    'rajasthan',
    'bihar',
    'assam',
    'madhya pradesh',
    'jharkhand',
    'gujarat',
    'tamil nadu',
    'karnataka'
)""").show(100,0)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

uuid,txn_amount,txn_count,cross_state_txn_amount,cross_state_txn_count,cp_state_count,dnm_txn_amount,dnm_txn_count,L1,reported,min_reporting_date,min_mha_txn,freeze_flag,combined_l1,signup_time,onboarding_date,onb_month,itc_flag,kyc_state,udyam_date_of_registration,udyam_to_onb,mcc,merchant_category,users,gst_ref_id,po_decile,po_score,o_decile,o_score,cp_state_5k_count,cp_state_10k_count,first_freeze_date,first_freeze_remark,last_freeze_remark
334b0c4e-500e-4cc6-86ec-f4858bf65db6,270342.28,76,258888.28,61,15,NULL,0,0,0,NULL,NULL,NULL,0,2026-06-01 06:24:45.76272,2026-06-01 00:00:00,6,RED,Rajasthan,2026-06-01,0,5968,SERVICES,1,NULL,NULL,NULL,NULL,NULL,8,5,2026-06-07 12:37:27.803,Fraud Rule Enigne : FraudCaseInvestigationCBSFreeze Requested:2 Final:2,Fraud Rule Enigne : FraudCaseInvestigationCBSFreeze Requested:2 Final:2


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

DataFrame[uuid: string, txn_amount: double, txn_count: bigint, 99_txn_amount: double, 99_txn_count: bigint, 999_txn_amount: double, 999_txn_count: bigint, 999_cp_count: bigint, mismatch_debit_amount: double, mismatch_debit_count: bigint, round_txn_amount: double, round_txn_count: bigint, decimal_txn_amount: double, decimal_txn_count: bigint, cross_state_txn_amount: double, cross_state_txn_count: bigint, cp_state_count: bigint, dnm_txn_amount: double, dnm_txn_count: bigint, L1: int, reported: int, min_reporting_date: string, min_mha_txn: string, freeze_flag: int, combined_l1: int, signup_time: timestamp, onboarding_date: timestamp, onb_month: int, itc_flag: string, uuid: string, kyc_state: string, udyam_date_of_registration: string, udyam_to_onb: int, mcc: string, merchant_category: string, users: int, gst_ref_id: bigint, po_decile: int, po_score: double, o_decile: bigint, o_score: double, cp_state_5k_count: bigint, cp_state_10k_count: bigint, first_freeze_date: timestamp, first_freeze_